# TruthLens AI — FEVER Dataset Deep-Dive EDA
This notebook performs comprehensive exploratory data analysis on the FEVER (Fact Extraction and VERification) dataset.

### Topics Covered:
- Label distribution (`SUPPORTS`, `REFUTES`, `NOT ENOUGH INFO`)
- Verifiable vs. Not Verifiable claims
- Claim length analysis (word and character distributions)
- Evidence structure (evidence sets, sentence pointers, Wikipedia page references)
- Annotation duplicates & contradictory labeling analysis

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath("../src"))
from fever_loader import load_fever_raw, parse_fever_evidence, get_fever_statistics

sns.set_theme(style="whitegrid")
df_fever, malformed = load_fever_raw("../data/raw/fever/train.jsonl")
print(f"Loaded FEVER train: {len(df_fever):,} rows")

## 1. Label Distribution & Verifiable Status

In [ ]:
lbl_counts = df_fever["label"].value_counts()
print(lbl_counts)

plt.figure(figsize=(7, 4.5))
sns.barplot(x=lbl_counts.index, y=lbl_counts.values, palette=["#2ca02c", "#7f7f7f", "#d62728"])
plt.title("FEVER Train Label Distribution")
plt.ylabel("Count")
plt.xlabel("Verification Label")
for i, v in enumerate(lbl_counts.values):
    plt.text(i, v / 2, f"{v:,}\n({v/len(df_fever)*100:.1f}%)", ha="center", va="center", color="white", fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Claim Length Distribution

In [ ]:
df_fever["word_count"] = df_fever["claim"].apply(lambda c: len(c.split()))
df_fever["char_count"] = df_fever["claim"].apply(len)

print(df_fever[["word_count", "char_count"]].describe())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df_fever["word_count"], bins=30, ax=ax1, color="#2b5c8f", kde=True)
ax1.set_title("Claim Word Count Distribution")
ax1.set_xlim(0, 35)

sns.boxplot(x=df_fever["label"], y=df_fever["word_count"], ax=ax2, palette="muted")
ax2.set_title("Claim Word Count by Label")
ax2.set_ylim(0, 35)
plt.tight_layout()
plt.show()

## 3. Evidence Structure Analysis
In FEVER, evidence is formatted as a 3-level list: `[Sets -> Sentences -> [annot_id, ev_id, wiki_page, sent_id]]`.

In [ ]:
ev_parsed = df_fever["evidence"].apply(parse_fever_evidence)
df_fever["num_ev_sents"] = ev_parsed.apply(lambda x: x["num_evidence_sentences"])
df_fever["num_ev_sets"] = ev_parsed.apply(lambda x: x["num_evidence_sets"])

print("Evidence sentences per claim:")
print(df_fever["num_ev_sents"].value_counts().head(8))

print(f"Total claims with at least 1 evidence reference: {(df_fever['num_ev_sents'] > 0).sum():,} ({(df_fever['num_ev_sents'] > 0).sum()/len(df_fever)*100:.1f}%)")
print(f"Total claims with 0 evidence references (NEI): {(df_fever['num_ev_sents'] == 0).sum():,} ({(df_fever['num_ev_sents'] == 0).sum()/len(df_fever)*100:.1f}%)")

## 4. Duplicate Claims & Label Conflicts
FEVER contains duplicate claim strings from distinct crowd-source annotators.

In [ ]:
unique_claims = df_fever["claim"].nunique()
print(f"Unique claims: {unique_claims:,} out of {len(df_fever):,} rows ({len(df_fever) - unique_claims:,} duplicates)")

# Conflicting annotations
claim_labels = df_fever.groupby("claim")["label"].nunique()
conflicting = claim_labels[claim_labels > 1]
print(f"Claims with conflicting annotations: {len(conflicting):,}")

# Show examples of conflicting annotations
sample_conflicts = conflicting.index[:3]
for c in sample_conflicts:
    print(f"Claim: {c}")
    display(df_fever[df_fever["claim"] == c][["id", "label", "evidence"]])